# **TRAINING MODEL**

In [1]:
import os
import csv
from datetime import datetime
import joblib
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
import case_config as cfg

# ============================================================
# TODO: FILES CONFIGURATION 
# ============================================================

TRAIN_FILE = os.path.join("data", "train.csv")
MODEL_FILE = r"model.joblib" # model output 

TRAIN_SIZE = 0.8
LOG_FILE = r"team3B_training_submit.csv"

# ============================================================
# NOTE: HYPERPARAMETERS TUING
# ============================================================
# Recommended tuning order:
#   1. max_depth
#   2. min_samples_leaf
#   3. min_samples_split
#   4. random_state
#   5. criterion

CRITERION = "entropy"
MAX_DEPTH = 8
MIN_SAMPLES_SPLIT = 18
MIN_SAMPLES_LEAF = 2
RANDOM_STATE = 42
CCP_ALPHA = 0.006

# ============================================================
# LOAD DATA
# ============================================================
def load_labelled(path):
    df = pd.read_csv(path)
    df = df.dropna(subset=cfg.FEATURES + ["y"])
    x = df[cfg.FEATURES].to_numpy(dtype=float)
    y = df["y"].astype(int).to_numpy()
    return df, x, y

def show_class_balance(df, title):
    print("")
    print("===== " + title + " =====")
    counts = df["y"].value_counts().sort_index()
    for class_id, n in counts.items():
        print(
            "  %d %-10s %5d  %5.1f%%"
            % (
                class_id,
                cfg.label_of(class_id),
                n,
                100.0 * n / len(df)
            )
        )

# ============================================================
# BUILD MODEL
# ============================================================

def build_model():
    return DecisionTreeClassifier(
        criterion=CRITERION,
        max_depth=MAX_DEPTH,
        min_samples_split=MIN_SAMPLES_SPLIT,
        min_samples_leaf=MIN_SAMPLES_LEAF,
        ccp_alpha=CCP_ALPHA,
        random_state= RANDOM_STATE,
    )
    
# ============================================================
# EVALUATE
# ============================================================

def evaluate(model, x, y, title):
    predicted = model.predict(x)
    score = f1_score(
        y,
        predicted,
        average="macro",
        labels=cfg.class_ids()
    )
    print("")
    print("===== " + title + " =====")
    print("Macro-F1: %.4f" % score)
    print(
        classification_report(
            y,
            predicted,
            labels=cfg.class_ids(),
            target_names=cfg.CLASS_LABELS,
            digits=4,
            zero_division=0
        )
    )
    print("Confusion matrix (rows = true, columns = predicted):")
    print(
        confusion_matrix(
            y,
            predicted,
            labels=cfg.class_ids()
        )
    )
    return score

def save_log(timestamp, criterion, max_depth, min_samples_split, min_samples_leaf, random_state, val_f1):
    file_exists = os.path.isfile(LOG_FILE)
    
    headers = [
        "timestamp", 
        "criterion", 
        "max_depth", 
        "min_samples_split", 
        "min_samples_leaf",
        "random_state", 
        "val_macro_f1"
    ]
    
    data = [
        timestamp, 
        criterion, 
        max_depth, 
        min_samples_split, 
        min_samples_leaf,
        random_state, 
        round(val_f1, 4)
    ]
    
    with open(LOG_FILE, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(headers)
        writer.writerow(data)
def main():
    print("Case:  " + cfg.CASE_TITLE)
    print("Model: " + cfg.MODEL_NAME)

    # NOTE: Load Train.csv
    train_df, x, y = load_labelled(TRAIN_FILE)
    print("")
    print("Total rows in train.csv: %d" % len(train_df))
    show_class_balance(train_df, "CLASS BALANCE: TRAINING")

    # NOTE: Split train.csv into TRAIN_SIZE train / validation
    x_train, x_val, y_train, y_val = train_test_split(
        x, y, train_size=TRAIN_SIZE, stratify=y, random_state=RANDOM_STATE
    )

    print("Training rows:   %d" % len(x_train))
    print("Validation rows: %d" % len(x_val))

    # NOTE: Train model using the training part
    model = build_model()
    model.fit(x_train, y_train)
    print("Model trained on %d%% training subset." % round(TRAIN_SIZE * 100))
    print(model)

    # NOTE: Evaluate using the validation part
    validation_f1 = evaluate(model, x_val, y_val, "VALIDATION SET")

    # NOTE: Final model using ALL train.csv
    final_model = build_model()
    final_model.fit(x,y)
    print("===== FINAL MODEL TRAINING =====")
    print("Final model trained on all %d rows in train.csv." % len(x))

    # NOTE: Save Final model
    joblib.dump(final_model, MODEL_FILE)
    print("Saved model: " + MODEL_FILE)
    print("Validation Macro-F1: %.4f" % validation_f1)

    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    save_log(
        timestamp=current_time,
        criterion=CRITERION,
        max_depth=MAX_DEPTH,
        min_samples_split=MIN_SAMPLES_SPLIT,
        min_samples_leaf=MIN_SAMPLES_LEAF,
        random_state= RANDOM_STATE,
        val_f1=validation_f1
    )
    print("Log saved to: " + LOG_FILE)
if __name__ == "__main__":
    main()

Case:  Greenhouse Environment State
Model: decision_tree

Total rows in train.csv: 613

===== CLASS BALANCE: TRAINING =====
  0 normal       378   61.7%
  1 alert        155   25.3%
  2 alarm         80   13.1%
Training rows:   490
Validation rows: 123
Model trained on 80% training subset.
DecisionTreeClassifier(ccp_alpha=0.006, criterion='entropy', max_depth=8,
                       min_samples_leaf=2, min_samples_split=18,
                       random_state=42)

===== VALIDATION SET =====
Macro-F1: 0.8770
              precision    recall  f1-score   support

      normal     0.9157    1.0000    0.9560        76
       alert     0.9167    0.7097    0.8000        31
       alarm     0.8750    0.8750    0.8750        16

    accuracy                         0.9106       123
   macro avg     0.9024    0.8616    0.8770       123
weighted avg     0.9106    0.9106    0.9061       123

Confusion matrix (rows = true, columns = predicted):
[[76  0  0]
 [ 7 22  2]
 [ 0  2 14]]
===== FINAL MO

# **TESTING MODEL**

In [2]:
import os
import csv
from datetime import datetime
import joblib
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import case_config as cfg

# ============================================================
# TODO: FILES CONFIGURATION 
# ============================================================
TEST_FILE = os.path.join("data", "test.csv")
MODEL_FILE = r"model.joblib" # same file cell 1 saves
LOG_FILE = r"team3B_testing_submit.csv"  

# ============================================================
# LOAD DATA
# ============================================================

def load_labelled(path):
    df = pd.read_csv(path)
    df = df.dropna(subset=cfg.FEATURES + ["y"])
    x = df[cfg.FEATURES].to_numpy(dtype=float)
    y = df["y"].astype(int).to_numpy()
    return df, x, y

def show_class_balance(df):
    print("")
    print("===== CLASS BALANCE: UNSEEN TEST SET =====")
    counts = df["y"].value_counts().sort_index()
    for class_id, n in counts.items():
        print(
            "  %d %-10s %5d  %5.1f%%"
            % (
                class_id,
                cfg.label_of(class_id),
                n,
                100.0 * n / len(df)
            )
        )

# ============================================================
# EVALUATE
# ============================================================

def evaluate(model, x, y):
    predicted = model.predict(x)
    score = f1_score(
        y,
        predicted,
        average="macro",
        labels=cfg.class_ids()
    )
    print("")
    print("===== FINAL UNSEEN TEST RESULT =====")
    print("Macro-F1: %.4f" % score)
    print(
        classification_report(
            y,
            predicted,
            labels=cfg.class_ids(),
            target_names=cfg.CLASS_LABELS,
            digits=4,
            zero_division=0
        )
    )
    print("Confusion matrix (rows = true, columns = predicted):")
    print(
        confusion_matrix(
            y,
            predicted,
            labels=cfg.class_ids()
        )
    )
    return score
# ============================================================
# FUNCTION FOR LOGGING TEST RESULTS
# ============================================================
def save_test_log(timestamp, model_file, params, test_f1):
    file_exists = os.path.isfile(LOG_FILE)
    
    headers = [
        "timestamp", 
        "model_file",
        "criterion", 
        "max_depth", 
        "min_samples_split", 
        "min_samples_leaf",
        "random_state", 
        "test_macro_f1"
    ]
    
    data = [
        timestamp,
        model_file,
        params.get("criterion"),
        params.get("max_depth"),
        params.get("min_samples_split"),
        params.get("min_samples_leaf"),
        params.get("random_state"),
        round(test_f1, 4)
    ]
    
    with open(LOG_FILE, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(headers)
        writer.writerow(data)
def main():
    print("Case:  " + cfg.CASE_TITLE)
    print("Model: " + cfg.MODEL_NAME)

    # NOTE: Load trained model
    model = joblib.load(MODEL_FILE)
    print("Loaded model: " + MODEL_FILE)

    # NOTE: Load test dataset
    test_df, x_test, y_test = load_labelled(TEST_FILE)
    print("Unseen test rows: %d" % len(test_df))
    show_class_balance(test_df)

    # NOTE: Final evaluation
    final_f1 = evaluate(
        model,
        x_test,
        y_test
    )
    print("Final unseen-test Macro-F1: %.4f" % final_f1)
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    params = model.get_params() 
    
    save_test_log(
        timestamp=current_time,
        model_file=MODEL_FILE,
        params=params,
        test_f1=final_f1
    )
    print("Test Log saved to: " + LOG_FILE)

if __name__ == "__main__":
    main()

Case:  Greenhouse Environment State
Model: decision_tree
Loaded model: model.joblib
Unseen test rows: 500

===== CLASS BALANCE: UNSEEN TEST SET =====
  0 normal       299   59.8%
  1 alert        118   23.6%
  2 alarm         83   16.6%

===== FINAL UNSEEN TEST RESULT =====
Macro-F1: 0.9106
              precision    recall  f1-score   support

      normal     0.9508    0.9699    0.9603       299
       alert     0.8596    0.8305    0.8448       118
       alarm     0.9383    0.9157    0.9268        83

    accuracy                         0.9280       500
   macro avg     0.9162    0.9054    0.9106       500
weighted avg     0.9272    0.9280    0.9275       500

Confusion matrix (rows = true, columns = predicted):
[[290   9   0]
 [ 15  98   5]
 [  0   7  76]]
Final unseen-test Macro-F1: 0.9106
Test Log saved to: team3B_testing_submit.csv
